# Description

In this notebook, I will explore the benchmark Human Eval and using GPT4 to generate it.

In [1]:
import os 
import sys
import numpy as np 
import pandas as pd 
import re
import io
import re
import time
import ast
import types
import unittest
import importlib
from typing import List, Tuple, Dict, Any, Set
import load_dotenv
from openai import OpenAI

# 1. Load data

In [2]:
PATH_CSV_DATA = "data/raw_data/human_eval.csv"

In [3]:
load_dotenv.load_dotenv()

# OPEN_AI_API = os.getenv("OPEN_AI_API")
OPEN_AI_API = os.getenv("OPEN_AI_API_v2")
if OPEN_AI_API is None:
    raise ValueError("OPEN_AI_API environment variable not set")
else:
    print("API key loaded successfully")

client = OpenAI(api_key=OPEN_AI_API)

API key loaded successfully


In [4]:
df = pd.read_csv(PATH_CSV_DATA)
print(f"Dataframe shape: {df.shape}")
df.sample(1)

Dataframe shape: (164, 5)


,task_id,prompt,canonical_solution,test,entry_point
149,HumanEval/149,"\ndef sorted_list_sum(lst):\n """"""Write a fu...",lst.sort()\n new_lst = []\n for i in...,def check(candidate):\n\n # Check some simp...,sorted_list_sum


In [5]:
idx = np.random.randint(0, df.shape[0])

code_description = df.loc[idx, "prompt"]
test_case = df.loc[idx, "test"]
entry_point = df.loc[idx, "entry_point"]

print("Code description:")
print(code_description)
print("=" * 20)
print("Test Case:")
print(test_case)

Code description:

def unique_digits(x):
    """Given a list of positive integers x. return a sorted list of all 
    elements that hasn't any even digit.

    Note: Returned list should be sorted in increasing order.
    
    For example:
    >>> unique_digits([15, 33, 1422, 1])
    [1, 15, 33]
    >>> unique_digits([152, 323, 1422, 10])
    []
    """

Test Case:
def check(candidate):

    # Check some simple cases
    assert candidate([15, 33, 1422, 1]) == [1, 15, 33]
    assert candidate([152, 323, 1422, 10]) == []
    assert candidate([12345, 2033, 111, 151]) == [111, 151]
    assert candidate([135, 103, 31]) == [31, 135]

    # Check some edge cases that are easy to work out by hand.
    assert True




# 2. Using GPT4 to generate sample

## 2.1. Generate code

In [6]:
def extract_function(llm_text):
    # 1) Grab text between <code>...</code>
    m = re.search(r"<code>\s*(.*?)\s*</code>", llm_text, flags=re.S|re.M)
    if not m:
        raise ValueError("No <code> block found")
    code = m.group(1)

    # 2) Optionally, if the model sometimes adds backticks, strip them
    code = re.sub(r"^```(?:python)?\s*|\s*```$", "", code.strip())

    return code

def extract_reasoning(llm_text):
    # 1) Grab text between <reasoning>...</reasoning>
    m = re.search(r"<reasoning>\s*(.*?)\s*</reasoning>", llm_text, flags=re.S|re.M)
    if not m:
        raise ValueError("No <reasoning> block found")
    reasoning = m.group(1).strip()
    return reasoning

In [7]:
constraints = """
Output only a complete and valid Python code for this function. Do not change the provided function signature.
Do NOT print any markdown or include the test cases in your output.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>
"""

COT = """
Before giving the final code, internally think step-by-step about how to implement the function
(do NOT print this reasoning).
"""

input_prompt = f"""write a complete python function
based on the following description:\n{code_description}.\n
{COT}.
with the following constraints:\n{constraints}
"""

print("Input prompt to GPT-4:")
print(input_prompt)

Input prompt to GPT-4:
write a complete python function
based on the following description:

def unique_digits(x):
    """Given a list of positive integers x. return a sorted list of all 
    elements that hasn't any even digit.

    Note: Returned list should be sorted in increasing order.
    
    For example:
    >>> unique_digits([15, 33, 1422, 1])
    [1, 15, 33]
    >>> unique_digits([152, 323, 1422, 10])
    []
    """
.


Before giving the final code, internally think step-by-step about how to implement the function
(do NOT print this reasoning).
.
with the following constraints:

Output only a complete and valid Python code for this function. Do not change the provided function signature.
Do NOT print any markdown or include the test cases in your output.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>




In [8]:
response = client.chat.completions.create(
    model="gpt-4o",  # or "gpt-4o" 
    messages=[
        {"role": "system", "content": "You are an expert in Python."},
        {"role": "user", "content": input_prompt},
    ],
)

output = response.choices[0].message.content

print("Response from OpenAI:")
print(output)

Response from OpenAI:
<code>
def unique_digits(x):
    """Given a list of positive integers x. return a sorted list of all 
    elements that hasn't any even digit.
    """
    def has_even_digit(number):
        # Check if a number contains any even digit
        while number > 0:
            digit = number % 10
            if digit % 2 == 0:
                return True
            number //= 10
        return False

    # Filter the input list to extract numbers with no even digits
    result = [num for num in x if not has_even_digit(num)]
    
    # Return the sorted list
    return sorted(result)
</code>


We can extract the complete code

In [9]:
completed_code = extract_function(output)
print(f"The complete code:\n")
print(completed_code)

The complete code:

def unique_digits(x):
    """Given a list of positive integers x. return a sorted list of all 
    elements that hasn't any even digit.
    """
    def has_even_digit(number):
        # Check if a number contains any even digit
        while number > 0:
            digit = number % 10
            if digit % 2 == 0:
                return True
            number //= 10
        return False

    # Filter the input list to extract numbers with no even digits
    result = [num for num in x if not has_even_digit(num)]
    
    # Return the sorted list
    return sorted(result)


## 2.2. Evaluate the generated code

In [10]:
import builtins
import typing

def create_namespace():
    ns = {}

    # 1. Standard builtins (print, len, etc.)
    ns.update({k: getattr(builtins, k) for k in dir(builtins)})

    # 2. Install common typing names (List, Optional, etc.)
    for name in typing.__all__:
        ns[name] = getattr(typing, name)

    # 3. (Optional) Add math, random, itertools, etc.
    import math, random, itertools, statistics
    ns.update({
        'math': math,
        'random': random,
        'itertools': itertools,
        'statistics': statistics,
    })

    return ns

In [11]:
def evaluate_asserts(generated_code: str, test_code: str, entry_point: str):
    # ns = {}
    ns = create_namespace()
    
    # 1. Exec both code strings
    exec(generated_code, ns)
    exec(test_code, ns)

    candidate = ns[entry_point]     # the model's function
    check_fn = ns["check"]          # original check() function
    
    # 2. Parse the test code AST
    tree = ast.parse(test_code)

    # 3. Find the check() function body
    check_body = None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == "check":
            check_body = node.body
            break

    if check_body is None:
        raise ValueError("check() function not found.")
    
    # 4. Evaluate each assert individually
    results = []
    for idx, stmt in enumerate(check_body):
        if isinstance(stmt, ast.Assert):
            # Convert AST back to executable code
            code = compile(ast.Module([stmt], type_ignores=[]), "<assert>", "exec")
            try:
                exec(code, {**ns, "candidate": candidate})
                results.append(("pass", None))
            except Exception as e:
                results.append(("fail", repr(e)))

    # 5. Compute pass percentage
    total = len(results)
    passed = sum(1 for r, _ in results if r == "pass")
    percentage = passed / total if total > 0 else 0.0

    return {
        "total_asserts": total,
        "passed": passed,
        "percentage": percentage,
        "detail": results
    }

In [12]:
result = evaluate_asserts(completed_code, test_case, entry_point)
print(result)

{'total_asserts': 5, 'passed': 5, 'percentage': 1.0, 'detail': [('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None)]}


# 3. Run through all sample

In [13]:
list_df = []

list_num_out_token = []

start_time = time.time()

for idx in range(df.shape[0]):
    if idx % 10 == 0:
        print(f"Processing idx={idx}/{df.shape[0]}")
    
    try:
        # 1. Prepare input prompt
        code_description = df.loc[idx, "prompt"]
        test_case = df.loc[idx, "test"]
        entry_point = df.loc[idx, "entry_point"]

        input_prompt = f"""write a complete python function
        based on the following description:\n{code_description}.\n
        {COT}.
        with the following constraints:\n{constraints}
        """

        # 2. Generate code with GPT-4o
        response = client.chat.completions.create(
            model="gpt-4o",  # or "gpt-4o" 
            messages=[
                {"role": "system", "content": "You are an expert in Python."},
                {"role": "user", "content": input_prompt},
            ],
        )

        output = response.choices[0].message.content
        completed_code = extract_function(output)
        num_out_tokens = response.usage.completion_tokens
        list_num_out_token.append(num_out_tokens)
        
        # 3. Evaluate the generated code
        result = evaluate_asserts(completed_code, test_case, entry_point)
        total_asserts = result["total_asserts"]
        passed_asserts = result["passed"]
        percentage = result["percentage"]
        
        list_df.append({
            "description": code_description,
            "generated_code": completed_code,
            "test_case": test_case,
            "entry_point": entry_point,
            "total_asserts": total_asserts,
            "passed_asserts": passed_asserts,
            "percentage": percentage
        })
    except Exception as e:
        print(f"Error at idx={idx}: {e}")
        continue
    
end_time = time.time()
avg_time_per_example = (end_time - start_time) / len(list_df)
print(f"Average time per example: {avg_time_per_example:.2f} seconds")

avg_output_tokens = sum(list_num_out_token) / len(list_num_out_token)
print(f"Average number of output tokens: {avg_output_tokens:.2f}")

Processing idx=0/164


Processing idx=10/164
Error at idx=10: No <code> block found
Processing idx=20/164
Processing idx=30/164
Processing idx=40/164
Processing idx=50/164
Processing idx=60/164
Processing idx=70/164
Processing idx=80/164
Processing idx=90/164
Processing idx=100/164
Processing idx=110/164
Processing idx=120/164
Processing idx=130/164
Processing idx=140/164
Processing idx=150/164
Processing idx=160/164
Average time per example: 1.50 seconds
Average number of output tokens: 118.72


In [14]:
output_df = pd.DataFrame(list_df)
print(f'Output dataframe shape: {output_df.shape}')
output_df.sample()

Output dataframe shape: (163, 7)


,description,generated_code,test_case,entry_point,total_asserts,passed_asserts,percentage
102,"\ndef rounded_avg(n, m):\n """"""You are given...","def rounded_avg(n, m):\n if n > m:\n ...",def check(candidate):\n\n # Check some simp...,rounded_avg,12,12,1.0


In [15]:
# # Save to CSV
# output_df.to_csv("data/human_eval_generated_gpt4o_COT.csv", index=False)

## 3.1. Check generated code

In [16]:
average_percentage = output_df["percentage"].mean()
print(f"Average pass percentage over all samples: {average_percentage:.2%}")    

Average pass percentage over all samples: 96.07%
